# 🔤 Byte-Pair Encoding (BPE) From Scratch

> Implementing the tokenization algorithm used in GPT, LLaMA, and modern LLMs

**What is BPE?**
- Start with individual characters as tokens
- Iteratively merge the most frequent adjacent pairs
- Continue until vocabulary reaches desired size

**Used by:** GPT-2, GPT-3, LLaMA, RoBERTa, and many more!

## 📚 Imports

In [ ]:
import re
from collections import defaultdict, Counter
import json
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports ready!")

## 💡 Part 1: Understanding BPE Intuition

**Example:**
```
Initial corpus (character-level):
l o w </w>
l o w e r </w>
n e w e s t </w>
w i d e s t </w>

Most frequent pair: ('e', 's') → merge → 'es'
Next: ('es', 't') → merge → 'est'
Next: ('l', 'o') → merge → 'lo'
Next: ('lo', 'w') → merge → 'low'

Final vocabulary: low, lower, newest, widest, ...
```

## 🔧 Part 2: BPE Tokenizer Implementation

In [ ]:
class BPETokenizer:
    """
    Byte-Pair Encoding Tokenizer implemented from scratch
    """
    
    def __init__(self, vocab_size=1000):
        """
        Args:
            vocab_size: Target vocabulary size (number of merge operations + initial chars)
        """
        self.vocab_size = vocab_size
        self.word_freqs = {}  # Word frequencies from training
        self.splits = {}      # Current splits of each word
        self.merges = []      # List of merge operations
        self.vocab = set()    # Final vocabulary
        
    def _pre_tokenize(self, text):
        """
        Simple pre-tokenization: split by whitespace and punctuation
        In practice, GPT-2 uses regex: "\\s+"
        """
        # Split on whitespace and keep punctuation separate
        # This is a simplified version
        words = re.findall(r"\w+|[^\w\s]", text.lower())
        return words
    
    def _get_word_freqs(self, text):
        """
        Get word frequencies from text
        """
        words = self._pre_tokenize(text)
        self.word_freqs = Counter(words)
        
        # Initialize splits: each word as list of characters + end token
        self.splits = {
            word: list(word) + ["</w>"]
            for word in self.word_freqs.keys()
        }
        
        # Build initial vocabulary from characters
        for word in self.word_freqs.keys():
            for char in word:
                self.vocab.add(char)
        self.vocab.add("</w>")
        
        print(f"✅ Pre-tokenization complete")
        print(f"   Unique words: {len(self.word_freqs)}")
        print(f"   Initial vocab size: {len(self.vocab)}")
    
    def _get_pair_freqs(self):
        """
        Count frequency of all adjacent symbol pairs
        """
        pair_freqs = defaultdict(int)
        
        for word, freq in self.word_freqs.items():
            split = self.splits[word]
            
            if len(split) == 1:
                continue
            
            for i in range(len(split) - 1):
                pair = (split[i], split[i + 1])
                pair_freqs[pair] += freq
        
        return pair_freqs
    
    def _merge_pair(self, pair):
        """
        Merge all occurrences of a pair in all words
        """
        merged_symbol = pair[0] + pair[1]
        
        for word in self.word_freqs.keys():
            split = self.splits[word]
            new_split = []
            i = 0
            
            while i < len(split):
                # Check if current and next symbol form the pair
                if i < len(split) - 1 and split[i] == pair[0] and split[i + 1] == pair[1]:
                    new_split.append(merged_symbol)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            
            self.splits[word] = new_split
        
        # Add merged symbol to vocabulary
        self.vocab.add(merged_symbol)
        
        return merged_symbol
    
    def train(self, text, verbose=True):
        """
        Train BPE on text
        
        Args:
            text: Training text
            verbose: Print progress
        """
        print("🚀 Starting BPE training...\n")
        
        # Step 1: Pre-tokenize and get frequencies
        self._get_word_freqs(text)
        
        # Step 2: Perform merges
        num_merges = self.vocab_size - len(self.vocab)
        
        if verbose:
            print(f"\n📊 Training config:")
            print(f"   Target vocab size: {self.vocab_size}")
            print(f"   Initial vocab size: {len(self.vocab)}")
            print(f"   Number of merges: {num_merges}\n")
        
        for i in range(num_merges):
            # Find most frequent pair
            pair_freqs = self._get_pair_freqs()
            
            if not pair_freqs:
                print(f"\n✅ No more pairs to merge at step {i}")
                break
            
            best_pair = max(pair_freqs, key=pair_freqs.get)
            best_freq = pair_freqs[best_pair]
            
            # Merge the pair
            merged = self._merge_pair(best_pair)
            self.merges.append(best_pair)
            
            if verbose and (i < 5 or i % 50 == 0):
                print(f"Merge {i+1:3d}: {best_pair[0]!r} + {best_pair[1]!r} → {merged!r} (freq: {best_freq})")
        
        print(f"\n✅ Training complete!")
        print(f"   Final vocab size: {len(self.vocab)}")
        print(f"   Total merges: {len(self.merges)}")
    
    def encode(self, text):
        """
        Encode text to token IDs
        """
        words = self._pre_tokenize(text)
        tokens = []
        
        for word in words:
            # Start with character-level split
            word_tokens = list(word) + ["</w>"]
            
            # Apply merges in order
            for merge in self.merges:
                new_tokens = []
                i = 0
                while i < len(word_tokens):
                    if (i < len(word_tokens) - 1 and 
                        word_tokens[i] == merge[0] and 
                        word_tokens[i + 1] == merge[1]):
                        new_tokens.append(merge[0] + merge[1])
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            
            tokens.extend(word_tokens)
        
        return tokens
    
    def decode(self, tokens):
        """
        Decode tokens back to text
        """
        text = "".join(tokens)
        text = text.replace("</w>", " ")
        return text.strip()
    
    def get_vocab(self):
        """Get sorted vocabulary"""
        return sorted(self.vocab, key=len, reverse=True)
    
    def save(self, path):
        """Save tokenizer to file"""
        data = {
            "vocab_size": self.vocab_size,
            "merges": [list(m) for m in self.merges],
            "vocab": list(self.vocab)
        }
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"✅ Tokenizer saved to {path}")
    
    @classmethod
    def load(cls, path):
        """Load tokenizer from file"""
        with open(path, 'r') as f:
            data = json.load(f)
        
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(m) for m in data["merges"]]
        tokenizer.vocab = set(data["vocab"])
        
        print(f"✅ Tokenizer loaded from {path}")
        return tokenizer

print("✅ BPE Tokenizer class created!")

## 🧪 Part 3: Training on Sample Data

In [ ]:
# Sample training data
training_text = """
The quick brown fox jumps over the lazy dog.
The dog sleeps on the mat.
The cat sits on the bed.
A bird flies in the sky.
The sun shines bright today.
The moon glows at night.
Stars twinkle in the dark sky.
The wind blows through the trees.
Rain falls on the ground.
Snow covers the mountain.
The river flows to the sea.
Fish swim in the water.
Birds sing in the morning.
The cat chases the mouse.
The dog barks at the stranger.
Children play in the park.
The teacher writes on the board.
Students read their books.
The cook prepares delicious food.
The farmer grows fresh vegetables.
"""

# Create and train tokenizer
print("=== Training BPE Tokenizer ===\n")

tokenizer = BPETokenizer(vocab_size=200)
tokenizer.train(training_text, verbose=True)

print(f"\n📚 Vocabulary (first 30 tokens):")
vocab_list = tokenizer.get_vocab()[:30]
for i, token in enumerate(vocab_list, 1):
    print(f"   {i:2d}. {token!r}")

## 🔍 Part 4: Encoding & Decoding Examples

In [ ]:
# Test encoding
print("=== Encoding Examples ===\n")

test_sentences = [
    "The cat sleeps",
    "A bird flies",
    "The sun shines",
    "Children play in the park"
]

for sentence in test_sentences:
    tokens = tokenizer.encode(sentence)
    decoded = tokenizer.decode(tokens)
    
    print(f"Original: {sentence}")
    print(f"Tokens:   {tokens}")
    print(f"Decoded:  {decoded}")
    print(f"Token count: {len(tokens)}")
    print(f"Character count: {len(sentence)}")
    print(f"Compression: {len(tokens)/len(sentence):.2%}")
    print("-" * 50)

# Show merge rules
print("\n📋 First 20 Merge Rules:")
for i, merge in enumerate(tokenizer.merges[:20], 1):
    merged = merge[0] + merge[1]
    print(f"   {i:2d}. {merge[0]!r} + {merge[1]!r} → {merged!r}")

## 📊 Part 5: Analyzing Vocabulary Growth

In [ ]:
# Analyze vocabulary composition
print("=== Vocabulary Analysis ===\n")

vocab = tokenizer.get_vocab()

# Categorize by length
length_dist = Counter(len(token) for token in vocab)

print("Token Length Distribution:")
for length in sorted(length_dist.keys()):
    count = length_dist[length]
    bar = "█" * (count // 2)
    print(f"   Length {length:2d}: {count:3d} tokens {bar}")

# Show longest tokens
print("\n🏆 Longest Tokens (Top 15):")
longest = sorted(vocab, key=len, reverse=True)[:15]
for i, token in enumerate(longest, 1):
    print(f"   {i:2d}. {token!r} (length: {len(token)})")

# Show single characters
print("\n🔤 Single Character Tokens:")
single_chars = [t for t in vocab if len(t) == 1]
print(f"   Count: {len(single_chars)}")
print(f"   Tokens: {sorted(single_chars)}")

# Word endings
print("\n🔚 Common Word Endings (</w> patterns):")
endings = [t for t in vocab if t.endswith("</w>") and len(t) > 4]
for ending in sorted(endings, key=len, reverse=True)[:10]:
    print(f"   {ending!r}")

## 🔄 Part 6: Comparing Different Vocab Sizes

In [ ]:
# Compare different vocabulary sizes
print("=== Vocabulary Size Comparison ===\n")

vocab_sizes = [50, 100, 200, 500]
test_text = "The quick brown fox jumps over the lazy dog"

results = []

for size in vocab_sizes:
    print(f"Training with vocab_size={size}...")
    
    tok = BPETokenizer(vocab_size=size)
    tok.train(training_text, verbose=False)
    
    tokens = tok.encode(test_text)
    decoded = tok.decode(tokens)
    
    results.append({
        'size': size,
        'actual_vocab': len(tok.vocab),
        'num_tokens': len(tokens),
        'compression': len(tokens) / len(test_text)
        'tokens': tokens
    })
    
    print(f"   Vocab: {len(tok.vocab)}, Tokens: {len(tokens)}, Compression: {len(tokens)/len(test_text):.2%}\n")

# Display comparison
print("\n📊 Comparison Table:")
print(f"{'Vocab Size':<12} {'Actual Vocab':<14} {'Tokens':<8} {'Compression':<12}")
print("-" * 50)

for r in results:
    print(f"{r['size']:<12} {r['actual_vocab']:<14} {r['num_tokens']:<8} {r['compression']:<12.2%}")

# Show tokens for each size
print("\n🔍 Tokenization Comparison:")
print(f"Text: '{test_text}'\n")

for r in results:
    print(f"Vocab {r['size']}: {r['tokens']}")

## 🆚 Part 7: BPE vs Character vs Word Tokenization

In [ ]:
# Compare different tokenization approaches
print("=== Tokenization Comparison ===\n")

test_text = "The quick brown fox jumps over the lazy dog"

# Character-level
char_tokens = list(test_text)

# Word-level (simple split)
word_tokens = test_text.lower().split()

# BPE
bpe_tokens = tokenizer.encode(test_text)

print(f"Text: '{test_text}'\n")

print(f"Character-level ({len(char_tokens)} tokens):")
print(f"   {char_tokens[:20]}...\n")

print(f"Word-level ({len(word_tokens)} tokens):")
print(f"   {word_tokens}\n")

print(f"BPE ({len(bpe_tokens)} tokens):")
print(f"   {bpe_tokens}\n")

# Comparison table
print("📊 Comparison:")
print(f"{'Method':<15} {'Tokens':<8} {'Vocab Size':<12} {'Pros':<20}")
print("-" * 70)
print(f"{'Character':<15} {len(char_tokens):<8} {'~100':<12} {'No OOV, simple':<20}")
print(f"{'Word':<15} {len(word_tokens):<8} {'~10000+':<12} {'Meaningful units':<20}")
print(f"{'BPE':<15} {len(bpe_tokens):<8} {'~50000':<12} {'Balance of both':<20}")

print("\n✅ BPE gives the best balance!")

## 💾 Part 8: Save & Load Tokenizer

In [ ]:
# Save tokenizer
tokenizer.save("bpe_tokenizer.json")

# Load tokenizer
loaded_tokenizer = BPETokenizer.load("bpe_tokenizer.json")

# Test loaded tokenizer
test = "The cat sleeps on the mat"
original_tokens = tokenizer.encode(test)
loaded_tokens = loaded_tokenizer.encode(test)

print(f"\nOriginal tokenizer: {original_tokens}")
print(f"Loaded tokenizer:   {loaded_tokens}")
print(f"Match: {original_tokens == loaded_tokens}")

## 🎯 Part 9: Exercises

1. **Implement BPE dropout**: Randomly skip merges during training for robustness
2. **Add special tokens**: [PAD], [UNK], [BOS], [EOS] tokens
3. **Byte-level BPE**: Start with bytes instead of characters (like GPT-2)
4. **WordPiece algorithm**: Implement Google's WordPiece (used in BERT)
5. **Unigram tokenization**: Implement SentencePiece's unigram algorithm
6. **Compare with tiktoken**: Use OpenAI's tokenizer and compare outputs
7. **Train on larger corpus**: Use Wikipedia or books for better vocabulary
8. **Subword regularization**: Implement multiple segmentation paths

## 📚 Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **BPE Algorithm** | Merge most frequent pairs iteratively |
| **Vocabulary Size** | Trade-off: small = generalization, large = efficiency |
| **Subword Units** | Balance between characters and words |
| **Compression** | BPE reduces sequence length vs character-level |
| **OOV Handling** | Rare words become subword combinations |
| **Training Data** | Vocabulary reflects training corpus patterns |

**Real-world tokenizers:**
- **GPT-2/3/4**: tiktoken (BPE-based)
- **LLaMA**: SentencePiece (BPE-based)
- **BERT**: WordPiece
- **T5**: SentencePiece

**Next:** Try `02-pretraining.ipynb` to train on your tokenized data! 🚀